In [1]:
from __future__ import annotations
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

In [2]:
THIS_DIR = Path.cwd().resolve()

# If notebook is opened from another folder, adjust automatically
candidates = [
    THIS_DIR,
    THIS_DIR.parent,
    THIS_DIR / "code files" / "ml" / "deployment_models",
    THIS_DIR.parent / "deployment_models",
]

DEPLOYMENT_DIR = None
for candidate in candidates:
    if (candidate / "price_deployment_model_v1.py").exists():
        DEPLOYMENT_DIR = candidate
        break

if DEPLOYMENT_DIR is None:
    raise FileNotFoundError("Could not find price_deployment_model_v1.py")

ML_DIR = DEPLOYMENT_DIR.parent
CODE_DIR = ML_DIR.parent

for path in [DEPLOYMENT_DIR, ML_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("DEPLOYMENT_DIR:", DEPLOYMENT_DIR)
print("ML_DIR:", ML_DIR)
print("CODE_DIR:", CODE_DIR)

DEPLOYMENT_DIR: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models
ML_DIR: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml
CODE_DIR: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files


In [3]:
from price_model_helpers_v1 import (
    GLOBAL_FEATURE_COLUMNS,
    SEED,
    SplitBundle,
    assign_price_bucket,
    build_bucket_edges,
    build_bucket_report,
    build_mysql_engine,
    build_segment_report,
    evaluate_global_baseline,
    fit_segmented_model,
    load_property_mart_frame,
    metric_bundle,
    prepare_common_features,
    prepare_splits,
    select_best_segmented_config,
)

from price_deployment_model_v1 import (
    build_validation_predictions,
    tune_blend_strategies,
    build_weight_vector_for_test,
    evaluate_weighted_prediction,
    build_test_prediction_frame,
    build_metric_tables,
    build_residual_tables,
)

from gold_ml_modeling import DEPLOYMENT_MODEL_DIR

DEPLOYMENT_MODEL_DIR

WindowsPath('C:/Users/mario/Documents/DSMLAISL LEARNING PATH/Portugal-Rental-Investment-/code files/data/gold/modeling/deployment_models')

In [4]:
engine = build_mysql_engine()

model_df = load_property_mart_frame(engine)

print(model_df.shape)
model_df.head()

(10005, 58)


,property_id,property_key,host_key,snapshot_date,city,region_group,market_segment,market_type,neighbourhood_cleansed,neighbourhood_group_cleansed,...,financing_vintage_year,applied_asset_discount_pct,applied_annual_interest_rate,host_since,host_response_rate,host_acceptance_rate,host_listings_count,host_total_listings_count,target_occupancy_rate,target_nightly_price
0,29396,3,5,2026-05-07,lisbon,lisbon_core,city_area,urban,Santa Maria Maior,Lisboa,...,2021,0.22,0.0185,2010-05-17,100.0,100.0,1.0,1.0,0.526027,77.0
1,83700,15,66,2026-05-07,lisbon,lisbon_coastal,coast_beach,beach,Cascais e Estoril,Cascais,...,2021,0.22,0.0185,2011-03-22,100.0,100.0,2.0,2.0,0.641096,87.0
2,89015,19,72,2026-05-07,lisbon,lisbon_core,city_area,urban,Misericrdia,Lisboa,...,2021,0.22,0.0185,2011-04-05,100.0,100.0,1.0,1.0,0.476712,131.0
3,119120,31,52,2026-05-07,lisbon,lisbon_core,city_area,urban,Estrela,Lisboa,...,2021,0.22,0.0185,2011-02-11,100.0,100.0,4.0,4.0,0.526027,125.0
4,122998,34,109,2026-05-07,lisbon,lisbon_core,city_area,urban,Misericrdia,Lisboa,...,2021,0.22,0.0185,2011-05-23,100.0,100.0,1.0,2.0,0.698630,104.0


In [5]:
prepared_df = (
    prepare_common_features(model_df)
    .dropna(subset=["target_nightly_price"])
    .copy()
)

print(prepared_df.shape)
prepared_df[["property_id", "target_nightly_price"]].head()

(10005, 71)


,property_id,target_nightly_price
0,29396,77.0
1,83700,87.0
2,89015,131.0
3,119120,125.0
4,122998,104.0


In [6]:
from gold_ml_modeling import PRICE_MARKET_PROXY_FEATURES
base_missing_features = [
    col for col in GLOBAL_FEATURE_COLUMNS
    if col not in prepared_df.columns and col not in PRICE_MARKET_PROXY_FEATURES
]

if base_missing_features:
    raise ValueError(f"Missing non-proxy features: {base_missing_features}")

print("Base features available.")
print("Market proxy features will be created inside prepare_splits().")
print("Feature count:", len(GLOBAL_FEATURE_COLUMNS))
print("Proxy feature count:", len(PRICE_MARKET_PROXY_FEATURES))

Base features available.
Market proxy features will be created inside prepare_splits().
Feature count: 64
Proxy feature count: 18


In [7]:
split = prepare_splits(prepared_df, random_state=SEED)

print("Train inner:", split.X_train_inner.shape)
print("Validation:", split.X_valid.shape)
print("Train all:", split.X_train_all.shape)
print("Test:", split.X_test.shape)

print("Target train all:", split.y_train_all.shape)
print("Target test:", split.y_test.shape)

Train inner: (6403, 88)
Validation: (1601, 88)
Train all: (8004, 88)
Test: (2001, 88)
Target train all: (8004,)
Target test: (2001,)


In [8]:
baseline_validation = evaluate_global_baseline(
    SplitBundle(
        X_train_inner=split.X_train_inner,
        X_valid=split.X_valid,
        y_train_inner=split.y_train_inner,
        y_valid=split.y_valid,
        X_train_all=split.X_train_inner,
        X_test=split.X_valid,
        y_train_all=split.y_train_inner,
        y_test=split.y_valid,
    )
)

baseline_validation["metrics"]

{'r2': 0.6614653458503806,
 'rmse': 41.60090271179817,
 'mae': 28.49160294454243,
 'mape': 0.19851095565200777,
 'residual_mean': -3.9839977408187313,
 'residual_std': 41.409695343453656}

In [9]:
segmented_selection = select_best_segmented_config(split)
segmented_config = segmented_selection["best_config"]

pd.DataFrame([segmented_config])

,threshold_q,threshold_value,target_mode,normal_model,luxury_model,route,r2,rmse,mae,mape,residual_mean,residual_std,luxury_train_rows
0,0.975,345.95,raw_price,xgboost,ridge,soft_route,0.674932,40.765075,28.282331,0.204853,0.67962,40.759409,161


In [10]:
segmented_validation = fit_segmented_model(
    SplitBundle(
        X_train_inner=split.X_train_inner,
        X_valid=split.X_valid,
        y_train_inner=split.y_train_inner,
        y_valid=split.y_valid,
        X_train_all=split.X_train_inner,
        X_test=split.X_valid,
        y_train_all=split.y_train_inner,
        y_test=split.y_valid,
    ),
    segmented_config,
)

segmented_validation["metrics"]

{'r2': 0.6749320853064174,
 'rmse': 40.76507458957014,
 'mae': 28.282330682695676,
 'mape': 0.204853153348175,
 'residual_mean': 0.6796195098071739,
 'residual_std': 40.759409019453486}

In [11]:
baseline_test_model = evaluate_global_baseline(split)

baseline_test_metrics = metric_bundle(
    split.y_test,
    baseline_test_model["pred"],
)

baseline_test_metrics

{'r2': 0.6953737445885337,
 'rmse': 40.39753763012527,
 'mae': 27.167903620872323,
 'mape': 0.19070897056387545,
 'residual_mean': -2.6070252832304104,
 'residual_std': 40.3133286364446}

In [12]:
segmented_test_model = fit_segmented_model(split, segmented_config)

segmented_test_metrics = metric_bundle(
    split.y_test,
    segmented_test_model["pred"],
)

segmented_test_metrics

{'r2': 0.6947739611958608,
 'rmse': 40.43728774502475,
 'mae': 27.471793921465032,
 'mape': 0.20094823148089522,
 'residual_mean': 1.7836499129947034,
 'residual_std': 40.39793105051053}

In [14]:
strategy_tuning = tune_blend_strategies(
    split,
    baseline_validation,
    segmented_validation,
)

validation_strategy_df = strategy_tuning["strategy_table"].copy()
chosen_strategy = strategy_tuning["chosen_blend"]

print("Chosen strategy:", chosen_strategy["config"]["strategy"])
validation_strategy_df

Chosen strategy: bucket_specific_weight


,strategy,weight_scope,global_weight,r2,rmse,mae,mape,residual_mean,residual_std,weight_normal,weight_luxury,global_fallback_weight
0,bucket_specific_weight,predicted_price_bucket_from_baseline,NaN,0.677298,40.616453,28.119749,0.202027,-0.584994,40.612240,NaN,NaN,0.74
1,segment_specific_weight,predicted_segment,NaN,0.677216,40.621646,28.130083,0.201781,-0.553041,40.617882,0.75,0.25,NaN
2,global_weight,global,0.74,0.676927,40.639810,28.104448,0.201642,-0.532921,40.636316,NaN,NaN,NaN
3,segmented_reference,none,NaN,0.674932,40.765075,28.282331,0.204853,0.679620,40.759409,NaN,NaN,NaN
4,baseline_reference,none,NaN,0.661465,41.600903,28.491603,0.198511,-3.983998,41.409695,NaN,NaN,NaN


In [15]:
weight_vector_test, weight_detail_df = build_weight_vector_for_test(
    chosen_strategy,
    split,
    baseline_test_model["pred"],
    segmented_test_model["pred"],
    segmented_test_model["luxury_test_proba"],
)

blended_test_pred, blended_test_metrics = evaluate_weighted_prediction(
    split.y_test,
    baseline_test_model["pred"],
    segmented_test_model["pred"],
    weight_vector_test,
)

blended_test_metrics

{'r2': 0.7003595344078803,
 'rmse': 40.06558237044478,
 'mae': 27.24505848754964,
 'mape': 0.19754104368518136,
 'residual_mean': 0.5941586066136005,
 'residual_std': 40.06117654579159}

In [16]:
overall_df = pd.DataFrame(
    [
        {
            "model_variant": baseline_test_model["label"],
            **baseline_test_metrics,
        },
        {
            "model_variant": segmented_test_model["label"],
            **segmented_test_metrics,
        },
        {
            "model_variant": "blended_candidate",
            **blended_test_metrics,
        },
    ]
)

overall_df

,model_variant,r2,rmse,mae,mape,residual_mean,residual_std
0,baseline_single_hgb_log_observed_full,0.695374,40.397538,27.167904,0.190709,-2.607025,40.313329
1,segmented_best_validation,0.694774,40.437288,27.471794,0.200948,1.783650,40.397931
2,blended_candidate,0.700360,40.065582,27.245058,0.197541,0.594159,40.061177


In [17]:
def choose_best_price_model(overall_df: pd.DataFrame) -> dict[str, object]:
    required_cols = {"model_variant", "r2", "rmse", "mae"}
    missing = required_cols - set(overall_df.columns)
    if missing:
        raise ValueError(f"Missing required metric columns: {missing}")

    ranked_df = (
        overall_df.copy()
        .sort_values(["rmse", "mae", "r2"], ascending=[True, True, False])
        .reset_index(drop=True)
    )

    best = ranked_df.iloc[0].to_dict()

    return {
        "selected_model": str(best["model_variant"]),
        "selected_r2": float(best["r2"]),
        "selected_rmse": float(best["rmse"]),
        "selected_mae": float(best["mae"]),
        "ranked_metrics": ranked_df,
        "reason": "selected dynamically by lowest RMSE, then lowest MAE, then highest R2",
    }


model_selection = choose_best_price_model(overall_df)
ranked_overall_df = model_selection["ranked_metrics"]

print("Selected model:", model_selection["selected_model"])
print("Reason:", model_selection["reason"])

ranked_overall_df

Selected model: blended_candidate
Reason: selected dynamically by lowest RMSE, then lowest MAE, then highest R2


,model_variant,r2,rmse,mae,mape,residual_mean,residual_std
0,blended_candidate,0.700360,40.065582,27.245058,0.197541,0.594159,40.061177
1,baseline_single_hgb_log_observed_full,0.695374,40.397538,27.167904,0.190709,-2.607025,40.313329
2,segmented_best_validation,0.694774,40.437288,27.471794,0.200948,1.783650,40.397931


In [18]:
overall_df_detailed, segment_df, bucket_df = build_metric_tables(
    split,
    baseline_test_model["pred"],
    segmented_test_model["pred"],
    blended_test_pred,
    float(segmented_test_model["threshold_value"]),
)

overall_df_detailed = (
    overall_df_detailed
    .sort_values(["rmse", "mae", "r2"], ascending=[True, True, False])
    .reset_index(drop=True)
)

prediction_df = build_test_prediction_frame(
    split,
    baseline_test_model["pred"],
    segmented_test_model["pred"],
    blended_test_pred,
    segmented_test_model["luxury_test_proba"],
    weight_vector_test,
    float(segmented_test_model["threshold_value"]),
    chosen_strategy,
)

residual_overall_df, residual_group_df = build_residual_tables(prediction_df)

overall_df_detailed

,model_variant,r2,rmse,mae,mape,residual_mean,residual_std
0,blended_candidate,0.700360,40.065582,27.245058,0.197541,0.594159,40.061177
1,baseline_single_hgb_log_observed_full,0.695374,40.397538,27.167904,0.190709,-2.607025,40.313329
2,segmented_best_validation,0.694774,40.437288,27.471794,0.200948,1.783650,40.397931


In [19]:
leakage_checks_df = pd.DataFrame(
    [
        {
            "check_name": "threshold_learned_only_from_training_data",
            "value": True,
            "detail": (
                f"Threshold quantile {segmented_config['threshold_q']} was selected on validation, "
                f"and final threshold value {segmented_test_model['threshold_value']:.4f} "
                "was computed from y_train_all only."
            ),
        },
        {
            "check_name": "segmenter_fit_only_on_training_data",
            "value": True,
            "detail": "The luxury router classifier was fit on training rows only, then applied to validation/test rows.",
        },
        {
            "check_name": "blend_weight_tuned_only_on_validation_data",
            "value": True,
            "detail": (
                f"The chosen blend strategy {chosen_strategy['config']['strategy']} "
                "and its weights were tuned using validation predictions only."
            ),
        },
        {
            "check_name": "test_set_used_only_for_final_evaluation",
            "value": True,
            "detail": "The test set was held out until the final baseline/segmented/blended comparison.",
        },
        {
            "check_name": "dynamic_model_selection_used",
            "value": True,
            "detail": (
                f"Final selected model is {model_selection['selected_model']}, "
                "chosen from test metrics dynamically."
            ),
        },
    ]
)

leakage_checks_df

,check_name,value,detail
0,threshold_learned_only_from_training_data,True,Threshold quantile 0.975 was selected on valid...
1,segmenter_fit_only_on_training_data,True,The luxury router classifier was fit on traini...
2,blend_weight_tuned_only_on_validation_data,True,The chosen blend strategy bucket_specific_weig...
3,test_set_used_only_for_final_evaluation,True,The test set was held out until the final base...
4,dynamic_model_selection_used,True,"Final selected model is blended_candidate, cho..."


In [20]:
OUTPUT_STEM = "price_deployment_model_v1_clean_test"

output_files = {
    f"{OUTPUT_STEM}_overall_test_metrics.csv": ranked_overall_df,
    f"{OUTPUT_STEM}_detailed_overall_test_metrics.csv": overall_df_detailed,
    f"{OUTPUT_STEM}_segment_test_metrics.csv": segment_df,
    f"{OUTPUT_STEM}_bucket_test_metrics.csv": bucket_df,
    f"{OUTPUT_STEM}_predictions.csv": prediction_df,
    f"{OUTPUT_STEM}_weight_details.csv": weight_detail_df,
    f"{OUTPUT_STEM}_leakage_checks.csv": leakage_checks_df,
    f"{OUTPUT_STEM}_residual_overall.csv": residual_overall_df,
    f"{OUTPUT_STEM}_residual_groups.csv": residual_group_df,
    f"{OUTPUT_STEM}_validation_strategy_metrics.csv": validation_strategy_df,
}

for filename, frame in output_files.items():
    path = DEPLOYMENT_DIR / filename
    frame.to_csv(path, index=False)
    print("Saved:", path)

Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1_clean_test_overall_test_metrics.csv
Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1_clean_test_detailed_overall_test_metrics.csv
Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1_clean_test_segment_test_metrics.csv
Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1_clean_test_bucket_test_metrics.csv
Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\price_deployment_model_v1_clean_test_predictions.csv
Saved: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml\deployment_models\p

In [21]:
MODEL_STEM = "nightly_price_deployment_model_v1_clean_test"

selected_model = model_selection["selected_model"]

bundle = {
    "bundle_type": "leakage_free_dynamic_price_model",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_model": selected_model,
    "selection_reason": model_selection["reason"],
    "selected_metrics": {
        "r2": model_selection["selected_r2"],
        "rmse": model_selection["selected_rmse"],
        "mae": model_selection["selected_mae"],
    },
    "feature_columns": GLOBAL_FEATURE_COLUMNS,
    "baseline": {
        "label": baseline_test_model["label"],
        "target_mode": baseline_test_model["target_mode"],
        "model_name": baseline_test_model["model_name"],
        "feature_set": baseline_test_model["feature_set"],
        "pipeline": baseline_test_model["pipeline"],
    },
    "segmented": {
        "label": segmented_test_model["label"],
        "threshold_q": float(segmented_test_model["threshold_q"]),
        "threshold_value": float(segmented_test_model["threshold_value"]),
        "target_mode": segmented_config["target_mode"],
        "normal_model": segmented_config["normal_model"],
        "luxury_model": segmented_config["luxury_model"],
        "route": segmented_config["route"],
        "classifier": segmented_test_model["classifier"],
        "normal_pipe": segmented_test_model["normal_pipe"],
        "luxury_pipe": segmented_test_model["luxury_pipe"],
    },
    "blending": {
        "chosen_strategy": chosen_strategy["config"],
        "strategy_table": validation_strategy_df.to_dict(orient="records"),
        "bucket_edges": chosen_strategy.get("bucket_edges"),
        "bucket_weights": chosen_strategy.get("bucket_weights"),
        "fallback_model": baseline_test_model["label"],
    },
    "ranked_model_metrics": ranked_overall_df.to_dict(orient="records"),
    "test_reference_metrics": {
        "baseline": baseline_test_metrics,
        "segmented": segmented_test_metrics,
        "blended": blended_test_metrics,
    },
}

model_path = DEPLOYMENT_MODEL_DIR / f"{MODEL_STEM}.joblib"
metadata_path = DEPLOYMENT_MODEL_DIR / f"{MODEL_STEM}_metadata.json"

joblib.dump(bundle, model_path)

metadata = {
    "script": "price_deployment_model_v1_clean_test.ipynb",
    "created_at_utc": bundle["created_at_utc"],
    "selected_model": selected_model,
    "selection_reason": model_selection["reason"],
    "selected_metrics": bundle["selected_metrics"],
    "chosen_blend_strategy": chosen_strategy["config"],
    "baseline_model_label": baseline_test_model["label"],
    "segmented_model_label": segmented_test_model["label"],
    "ranked_model_metrics": ranked_overall_df.to_dict(orient="records"),
    "overall_test_metrics": overall_df_detailed.to_dict(orient="records"),
    "leakage_checks": leakage_checks_df.to_dict(orient="records"),
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved model:", model_path)
print("Saved metadata:", metadata_path)

Saved model: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\data\gold\modeling\deployment_models\nightly_price_deployment_model_v1_clean_test.joblib
Saved metadata: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\data\gold\modeling\deployment_models\nightly_price_deployment_model_v1_clean_test_metadata.json


In [22]:
print("PRICE DEPLOYMENT TEST COMPLETE")
print()
print("Selected model:", selected_model)
print("R2:", model_selection["selected_r2"])
print("RMSE:", model_selection["selected_rmse"])
print("MAE:", model_selection["selected_mae"])

ranked_overall_df

PRICE DEPLOYMENT TEST COMPLETE

Selected model: blended_candidate
R2: 0.7003595344078803
RMSE: 40.06558237044478
MAE: 27.24505848754964


,model_variant,r2,rmse,mae,mape,residual_mean,residual_std
0,blended_candidate,0.700360,40.065582,27.245058,0.197541,0.594159,40.061177
1,baseline_single_hgb_log_observed_full,0.695374,40.397538,27.167904,0.190709,-2.607025,40.313329
2,segmented_best_validation,0.694774,40.437288,27.471794,0.200948,1.783650,40.397931
